# Shahnameh style fine-tune with Unsloth (Kaggle, 2x T4)

This notebook teaches a small model to answer in the voice of Ferdowsi's Shahnameh, in Persian, English and German.

## Before you press Run All
1. **Upload the data.** On Kaggle, create a new Dataset. Upload the 8 `.jsonl` files from the `training_data` folder (`fa_train.jsonl`, `all_val.jsonl`, and so on). You can give the dataset any name.
2. **Notebook settings** (right-hand panel):
   - Accelerator: **GPU T4 x2**
   - Internet: **On**. This needs a phone-verified Kaggle account.
3. **Add Input:** attach the dataset from step 1. The notebook finds the files itself.
4. **Optional:** to push the adapter to the Hugging Face Hub, go to *Add-ons, Secrets*, add `HF_TOKEN`, and set `HF_REPO` below.
5. Edit the **Settings** cell if you like, then **Run All**.

## What it does with the two GPUs
- `GPU_MODE = "train+baseline"` (default, most reliable): GPU 0 trains while GPU 1 writes answers from the untouched base model. The baseline is then ready for comparison.
- `GPU_MODE = "ddp"`: training runs on both GPUs with `torchrun`. Unsloth still calls multi-GPU training experimental, so use this only if the default works and you want it faster. The baseline runs first, on GPU 0.

## Outputs (in `/kaggle/working`)
| path | what |
|---|---|
| `shahnameh-lora/` | trained LoRA adapter + tokenizer + `train_log.json` |
| `shahnameh-lora.zip` | the same, zipped for download |
| `samples_baseline.jsonl`, `samples_finetuned.jsonl` | answers to the 30 validation prompts |
| `comparison.md` | baseline and fine-tuned answers side by side |

If the training loss becomes `nan` or stays at 0 with Gemma 4, switch `MODEL_PRESET` to `"qwen3-4b"` and restart the session. T4 GPUs only run float16.

## 1. Settings

In [ ]:
# ---------- Settings ----------
MODEL_PRESET = "qwen3-4b"      # "qwen3-4b" (default: strong Persian) or "gemma-4-e4b" (140 languages)
GPU_MODE = "train+baseline"    # "train+baseline" or "ddp" (see notes above)
LANGS = ["fa", "en", "de"]     # train on a subset, e.g. ["fa"]
USE_SYSTEM_PROMPT = True       # False = train without the system message

EPOCHS = 3                     # the best epoch (lowest validation loss) is kept automatically
LEARNING_RATE = 2e-4
LORA_R = 16
BATCH_SIZE = 2                 # per GPU; lower to 1 if you run out of memory
GRAD_ACCUM = 4
MAX_SEQ_LEN = 1024

EXPORT_MERGED_16BIT = False    # large; may not fit Kaggle's 20 GB output limit
EXPORT_GGUF = False            # for llama.cpp / Ollama; large and slow
HF_REPO = ""                   # e.g. "your-name/shahnameh-lora"; needs the HF_TOKEN secret

WORK = "/kaggle/working"
SCRIPTS = f"{WORK}/scripts"
ADAPTER = f"{WORK}/shahnameh-lora"

assert MODEL_PRESET in ("gemma-4-e4b", "qwen3-4b"), MODEL_PRESET
assert GPU_MODE in ("train+baseline", "ddp"), GPU_MODE
import os
os.makedirs(SCRIPTS, exist_ok=True)

## 2. Check the GPUs

In [ ]:
import subprocess
gpus = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
                      capture_output=True, text=True, check=True).stdout.strip().splitlines()
print("\n".join(gpus))
if len(gpus) < 2:
    raise SystemExit("Only %d GPU found. Set the accelerator to 'GPU T4 x2' in the notebook settings." % len(gpus))

## 3. Install Unsloth (a few minutes)

In [ ]:
# Pinned installs taken from Unsloth's own Kaggle notebooks for each model family.
if MODEL_PRESET == "gemma-4-e4b":
    !pip install -q unsloth
    !pip install -q --no-deps transformers==5.10.1 "tokenizers>=0.22.0,<=0.23.0"
    !pip install -q "huggingface_hub>=1.5.0,<2.0"
    !pip install -q torchcodec
    !pip install -q --no-deps --upgrade timm
else:
    !pip install -q pip3-autoremove
    !pip install -q torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
    !pip install -q unsloth
    !pip install -q --no-deps --upgrade "torchao>=0.16.0"
    !pip install -q transformers==4.56.2
    !pip install -q --no-deps trl==0.22.2

## 4. Write the training scripts
The scripts run as separate processes, which lets both GPUs work at once. Edit them here if you want to change something.

In [ ]:
%%writefile /kaggle/working/scripts/shahnameh_common.py
"""Shared helpers for the Shahnameh Unsloth scripts. Standard library only, so it can be imported before unsloth."""
import json
import re
from pathlib import Path

LANGS = ("fa", "en", "de")
THINK_BLOCK = re.compile(r"^\s*<think>.*?</think>\s*", re.DOTALL)

# Repetition control is model-dependent; measure before assuming.
#  - SmolLM2-360M NEEDED it: greedy looped forever, 0/30 held-out prompts
#    terminated, vs 25/30 with repetition_penalty=1.15.
#  - Qwen3-4B is HURT by it on Persian verse: masnavi rhyme requires recurring
#    sounds and function words, and penalising them breaks the rhyme. Measured
#    Persian rhyme rate: 39% with rp=1.1 vs 53% with none. Gemma keeps a mild
#    penalty as it is untested here.
MODEL_PRESETS = {
    "gemma-4-e4b": {
        "model_name": "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit",
        "loader": "FastModel",
        "chat_template": "gemma-4",
        "strip_prefix": "<bos>",
        "generation": {"temperature": 1.0, "top_p": 0.95, "top_k": 64, "repetition_penalty": 1.1},
        "gguf_quant": "Q8_0",  # Unsloth's Gemma 4 notebook: only Q8_0, BF16, F16 for now
    },
    "qwen3-4b": {
        "model_name": "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit",
        "loader": "FastLanguageModel",
        "chat_template": "qwen3-instruct",
        "strip_prefix": "",
        "generation": {"temperature": 0.7, "top_p": 0.8, "top_k": 20},  # no repetition_penalty: see note
        "gguf_quant": "q4_k_m",
    },
}


def get_preset(name):
    if name not in MODEL_PRESETS:
        raise ValueError(f"unknown model preset {name!r}; choose one of {sorted(MODEL_PRESETS)}")
    return MODEL_PRESETS[name]


def find_data_dir(root):
    """Return the folder under `root` that holds all_train.jsonl (Kaggle mounts datasets at unknown depths)."""
    matches = sorted(Path(root).rglob("all_train.jsonl"))
    if not matches:
        raise FileNotFoundError(f"no all_train.jsonl found under {root}; did you add the dataset to the notebook?")
    return matches[0].parent


def load_examples(data_dir, split, langs=LANGS, use_system=True):
    """Load chat examples for the chosen languages, tagging each with its language."""
    unknown = set(langs) - set(LANGS)
    if unknown:
        raise ValueError(f"unknown languages {sorted(unknown)}")
    examples = []
    for lang in langs:
        path = Path(data_dir) / f"{lang}_{split}.jsonl"
        if not path.exists():
            raise FileNotFoundError(f"missing {path}")
        for line_no, line in enumerate(path.read_text(encoding="utf-8").splitlines(), start=1):
            if not line.strip():
                continue
            messages = json.loads(line)["messages"]
            validate_messages(messages, f"{path.name}:{line_no}")
            kept = [m for m in messages if use_system or m["role"] != "system"]
            examples.append({"lang": lang, "messages": kept})
    return examples


def custom_example(data_dir, lang, prompt, use_system=True):
    """A one-off question in the same shape as the data, reusing that language's system prompt."""
    template = load_examples(data_dir, "val", [lang], use_system)[0]["messages"]
    system = [m for m in template if m["role"] == "system"]
    return {"lang": lang, "messages": system + [{"role": "user", "content": prompt},
                                                 {"role": "assistant", "content": "(generated)"}]}


def validate_messages(messages, where):
    roles = [m.get("role") for m in messages]
    if roles[-2:] != ["user", "assistant"] or any(not m.get("content") for m in messages):
        raise ValueError(f"{where}: expected ...user, assistant with non-empty content, got roles {roles}")


def render_chat(tokenizer, messages, add_generation_prompt, strip_prefix=""):
    """Apply the chat template; fold the system prompt into the user turn if the template rejects system roles."""
    kwargs = {"tokenize": False, "add_generation_prompt": add_generation_prompt}
    try:
        text = tokenizer.apply_chat_template(messages, **kwargs)
    except Exception as err:  # jinja2 TemplateError differs per template; only retry when a system turn exists
        if not messages or messages[0]["role"] != "system":
            raise
        text = tokenizer.apply_chat_template(fold_system_prompt(messages), **kwargs)
        print(f"note: chat template rejected the system role ({err}); folded it into the user turn")
    return text[len(strip_prefix):] if strip_prefix and text.startswith(strip_prefix) else text


def generation_prompt(tokenizer, messages, preset):
    """Prompt text for generation; models that expect a BOS prefix always get exactly one."""
    text = render_chat(tokenizer, messages, True)
    prefix = preset["strip_prefix"]
    return text if not prefix or text.startswith(prefix) else prefix + text


def fold_system_prompt(messages):
    system, first_user, *rest = messages
    merged = {"role": "user", "content": f"{system['content']}\n\n{first_user['content']}"}
    return [merged, *rest]


def clean_answer(text):
    """Drop an (empty) <think> block that some Qwen templates put at the start of replies."""
    return THINK_BLOCK.sub("", text).strip()


def checkpoint_dir(adapter_dir):
    """Checkpoints live next to the adapter folder, so zipping the adapter stays small."""
    path = Path(adapter_dir)
    return path.parent / f"{path.name}-checkpoints"


def squash(text):
    return "".join(text.split())


def check_response_mask(trained_text, messages):
    """Raise if the tokens that receive loss are not the assistant reply (catches a broken response mask)."""
    answer = squash(messages[-1]["content"])
    prompt = squash(messages[-2]["content"])
    trained = squash(trained_text)
    if not trained:
        raise AssertionError("no tokens are trained: the response mask removed everything")
    if answer[:20] not in trained:
        raise AssertionError(f"assistant reply is not in the trained tokens: {trained_text[:200]!r}")
    if prompt and prompt in trained:
        raise AssertionError(f"user prompt leaked into the trained tokens: {trained_text[:200]!r}")


def write_jsonl(path, rows):
    Path(path).write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")


def read_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]


def comparison_markdown(baseline_rows, finetuned_rows):
    """Side-by-side markdown of baseline vs fine-tuned answers, matched by prompt."""
    finetuned = {(r["lang"], r["prompt"]): r["answer"] for r in finetuned_rows}
    parts = ["# Baseline vs fine-tuned\n"]
    for row in baseline_rows:
        key = (row["lang"], row["prompt"])
        parts.append(f"## [{row['lang']}] {row['prompt']}\n")
        parts.append("**Baseline**\n\n" + fence(row["answer"]))
        parts.append("**Fine-tuned**\n\n" + fence(finetuned.get(key, "(missing)")))
    return "\n".join(parts)


def fence(text):
    return "```text\n" + text.strip() + "\n```\n"


In [ ]:
%%writefile /kaggle/working/scripts/train_shahnameh.py
"""LoRA fine-tune on the Shahnameh style data with Unsloth.

One GPU:  CUDA_VISIBLE_DEVICES=0 python train_shahnameh.py --preset gemma-4-e4b
Two GPUs: torchrun --nproc_per_node=2 train_shahnameh.py --preset gemma-4-e4b   (DDP, still experimental in Unsloth)
"""
import unsloth  # noqa: F401  must be imported before transformers / trl
import argparse
import json
import os
from pathlib import Path

import torch
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel, FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

import shahnameh_common as common

torch._dynamo.config.recompile_limit = 64


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--preset", default="gemma-4-e4b", choices=sorted(common.MODEL_PRESETS))
    p.add_argument("--data-root", default="/kaggle/input")
    p.add_argument("--out", default="/kaggle/working/shahnameh-lora")
    p.add_argument("--langs", nargs="+", default=list(common.LANGS), choices=common.LANGS)
    p.add_argument("--no-system", action="store_true")
    p.add_argument("--epochs", type=float, default=3)
    p.add_argument("--lr", type=float, default=2e-4)
    p.add_argument("--lora-r", type=int, default=16)
    p.add_argument("--batch-size", type=int, default=2)
    p.add_argument("--grad-accum", type=int, default=4)
    p.add_argument("--max-seq-len", type=int, default=1024)
    p.add_argument("--seed", type=int, default=3407)
    return p.parse_args()


def load_model(preset, args, local_rank, world_size):
    loader = FastModel if preset["loader"] == "FastModel" else FastLanguageModel
    extra = {"device_map": {"": local_rank}} if world_size > 1 else {}
    model, tokenizer = loader.from_pretrained(
        model_name=preset["model_name"], max_seq_length=args.max_seq_len,
        load_in_4bit=True, dtype=None, full_finetuning=False, **extra,
    )
    if preset["loader"] == "FastModel":
        model = FastModel.get_peft_model(
            model, finetune_vision_layers=False, finetune_language_layers=True,
            finetune_attention_modules=True, finetune_mlp_modules=True,
            r=args.lora_r, lora_alpha=args.lora_r, lora_dropout=0, bias="none", random_state=args.seed,
        )
    else:
        model = FastLanguageModel.get_peft_model(
            model, r=args.lora_r, lora_alpha=args.lora_r, lora_dropout=0, bias="none",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            use_gradient_checkpointing="unsloth", random_state=args.seed,
        )
    return model, get_chat_template(tokenizer, chat_template=preset["chat_template"])


def to_dataset(examples, tokenizer, preset):
    texts = [common.render_chat(tokenizer, ex["messages"], False, preset["strip_prefix"]) for ex in examples]
    return Dataset.from_dict({"text": texts})


def verify_masking(trainer, tokenizer, examples, is_main):
    """Decode the tokens that get loss for the first rows and make sure they are only the assistant replies."""
    text_tok = getattr(tokenizer, "tokenizer", tokenizer)
    for i in range(min(3, len(examples))):
        row = trainer.train_dataset[i]
        trained_ids = [t for t, label in zip(row["input_ids"], row["labels"]) if label != -100]
        trained_text = text_tok.decode(trained_ids, skip_special_tokens=True)
        common.check_response_mask(trained_text, examples[i]["messages"])
        if is_main and i == 0:
            print("mask check ok. trained tokens of row 0:\n" + trained_text[:400])


def main():
    args = parse_args()
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    world_size = int(os.environ.get("WORLD_SIZE", 1))
    is_main = local_rank == 0
    preset = common.get_preset(args.preset)
    data_dir = common.find_data_dir(args.data_root)
    train_examples = common.load_examples(data_dir, "train", args.langs, not args.no_system)
    val_examples = common.load_examples(data_dir, "val", args.langs, not args.no_system)
    if is_main:
        print(f"data: {data_dir} | train {len(train_examples)} | val {len(val_examples)} | gpus {world_size}")

    model, tokenizer = load_model(preset, args, local_rank, world_size)
    use_bf16 = torch.cuda.is_bf16_supported()
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=to_dataset(train_examples, tokenizer, preset),
        eval_dataset=to_dataset(val_examples, tokenizer, preset),
        args=SFTConfig(
            dataset_text_field="text",
            output_dir=str(common.checkpoint_dir(args.out)),
            per_device_train_batch_size=args.batch_size,
            per_device_eval_batch_size=args.batch_size,
            gradient_accumulation_steps=args.grad_accum,
            num_train_epochs=args.epochs,
            learning_rate=args.lr,
            warmup_steps=5,
            lr_scheduler_type="linear",
            optim="adamw_8bit",
            weight_decay=0.001,
            logging_steps=1,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            fp16=not use_bf16,
            bf16=use_bf16,
            ddp_find_unused_parameters=False if world_size > 1 else None,
            seed=args.seed,
            report_to="none",
        ),
    )
    trainer = train_on_responses_only(trainer)
    verify_masking(trainer, tokenizer, train_examples, is_main)

    stats = trainer.train()
    if trainer.is_world_process_zero():
        model.save_pretrained(args.out)
        tokenizer.save_pretrained(args.out)
        log = {"preset": args.preset, "args": vars(args), "metrics": stats.metrics,
               "best_checkpoint": trainer.state.best_model_checkpoint,
               "best_eval_loss": trainer.state.best_metric, "log_history": trainer.state.log_history}
        Path(args.out, "train_log.json").write_text(json.dumps(log, indent=2), encoding="utf-8")
        print(f"saved LoRA adapter to {args.out} | best eval_loss {trainer.state.best_metric}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/scripts/sample_shahnameh.py
"""Generate answers for the validation prompts, from the base model or from a trained LoRA adapter.

Baseline:   CUDA_VISIBLE_DEVICES=1 python sample_shahnameh.py --preset gemma-4-e4b --out samples_baseline.jsonl
Fine-tuned: CUDA_VISIBLE_DEVICES=0 python sample_shahnameh.py --preset gemma-4-e4b --adapter /kaggle/working/shahnameh-lora --out samples_finetuned.jsonl
"""
import unsloth  # noqa: F401  must be imported before transformers
import argparse
import time

import torch
from unsloth import FastLanguageModel, FastModel
from unsloth.chat_templates import get_chat_template

import shahnameh_common as common

torch._dynamo.config.recompile_limit = 64


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--preset", default="gemma-4-e4b", choices=sorted(common.MODEL_PRESETS))
    p.add_argument("--adapter", default=None, help="trained LoRA folder; omit for the base model")
    p.add_argument("--data-root", default="/kaggle/input")
    p.add_argument("--out", required=True)
    p.add_argument("--prompt", default=None, help="answer this one question instead of the validation set")
    p.add_argument("--langs", nargs="+", default=list(common.LANGS), choices=common.LANGS)
    p.add_argument("--no-system", action="store_true")
    p.add_argument("--max-new-tokens", type=int, default=512)
    p.add_argument("--batch-size", type=int, default=6)
    p.add_argument("--max-seq-len", type=int, default=1024)
    return p.parse_args()


def main():
    args = parse_args()
    preset = common.get_preset(args.preset)
    data_dir = common.find_data_dir(args.data_root)
    if args.prompt:
        examples = [common.custom_example(data_dir, args.langs[0], args.prompt, not args.no_system)]
    else:
        examples = common.load_examples(data_dir, "val", args.langs, not args.no_system)
    loader = FastModel if preset["loader"] == "FastModel" else FastLanguageModel
    model, tokenizer = loader.from_pretrained(
        model_name=args.adapter or preset["model_name"], max_seq_length=args.max_seq_len,
        load_in_4bit=True, dtype=None,
    )
    tokenizer = get_chat_template(tokenizer, chat_template=preset["chat_template"])
    if hasattr(loader, "for_inference"):
        loader.for_inference(model)
    text_tok = getattr(tokenizer, "tokenizer", tokenizer)
    text_tok.padding_side = "left"
    if text_tok.pad_token is None:
        text_tok.pad_token = text_tok.eos_token

    rows, started = [], time.time()
    for start in range(0, len(examples), args.batch_size):
        batch = examples[start:start + args.batch_size]
        prompts = [common.generation_prompt(tokenizer, ex["messages"][:-1], preset) for ex in batch]
        inputs = text_tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=args.max_new_tokens, do_sample=True,
                                    pad_token_id=text_tok.pad_token_id, **preset["generation"])
        new_tokens = output[:, inputs["input_ids"].shape[1]:]
        for ex, answer in zip(batch, text_tok.batch_decode(new_tokens, skip_special_tokens=True)):
            rows.append({"lang": ex["lang"], "prompt": ex["messages"][-2]["content"],
                         "reference": ex["messages"][-1]["content"], "answer": common.clean_answer(answer)})
        print(f"{len(rows)}/{len(examples)} done ({time.time() - started:.0f}s)", flush=True)
    common.write_jsonl(args.out, rows)
    print(f"wrote {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/scripts/export_shahnameh.py
"""Optional exports of a trained adapter: merged 16-bit model, GGUF, and/or upload to the Hugging Face Hub.

CUDA_VISIBLE_DEVICES=0 HF_TOKEN=... python export_shahnameh.py --preset gemma-4-e4b --adapter /kaggle/working/shahnameh-lora --gguf --hf-repo you/shahnameh-lora
"""
import unsloth  # noqa: F401  must be imported before transformers
import argparse
import os

from unsloth import FastLanguageModel, FastModel

import shahnameh_common as common


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--preset", default="gemma-4-e4b", choices=sorted(common.MODEL_PRESETS))
    p.add_argument("--adapter", required=True)
    p.add_argument("--merged", action="store_true", help="save a merged 16-bit model (large)")
    p.add_argument("--gguf", action="store_true", help="save a GGUF file for llama.cpp / Ollama (large, slow)")
    p.add_argument("--hf-repo", default="", help="push the LoRA adapter to this Hub repo (needs HF_TOKEN)")
    p.add_argument("--out", default="/kaggle/working")
    return p.parse_args()


def main():
    args = parse_args()
    if not (args.merged or args.gguf or args.hf_repo):
        raise SystemExit("nothing to do: pass --merged, --gguf and/or --hf-repo")
    token = os.environ.get("HF_TOKEN", "")
    if args.hf_repo and not token:
        raise SystemExit("--hf-repo needs the HF_TOKEN environment variable")

    preset = common.get_preset(args.preset)
    loader = FastModel if preset["loader"] == "FastModel" else FastLanguageModel
    model, tokenizer = loader.from_pretrained(model_name=args.adapter, max_seq_length=1024, load_in_4bit=True)

    if args.hf_repo:
        model.push_to_hub(args.hf_repo, token=token)
        tokenizer.push_to_hub(args.hf_repo, token=token)
        print(f"pushed adapter to https://huggingface.co/{args.hf_repo}")
    if args.merged:
        target = os.path.join(args.out, "shahnameh-merged-16bit")
        model.save_pretrained_merged(target, tokenizer, save_method="merged_16bit")
        print(f"saved merged model to {target}")
    if args.gguf:
        target = os.path.join(args.out, "shahnameh-gguf")
        model.save_pretrained_gguf(target, tokenizer, quantization_method=preset["gguf_quant"])
        print(f"saved GGUF ({preset['gguf_quant']}) to {target}")


if __name__ == "__main__":
    main()


## 5. Find the data and download the model

In [ ]:
import os, sys, json, subprocess
sys.path.insert(0, SCRIPTS)
import shahnameh_common as common
from huggingface_hub import snapshot_download

DATA_DIR = common.find_data_dir("/kaggle/input")
preset = common.get_preset(MODEL_PRESET)
print("data:", DATA_DIR, sorted(p.name for p in DATA_DIR.glob("*.jsonl")))

# Download the model once, so two processes never fetch it at the same time.
snapshot_download(preset["model_name"])

DATA_ARGS = ["--preset", MODEL_PRESET, "--data-root", "/kaggle/input", "--langs", *LANGS]
if not USE_SYSTEM_PROMPT:
    DATA_ARGS.append("--no-system")


def run(cmd, gpus, name, background=False):
    """Run a script on the given GPUs in its own folder (Unsloth writes a compile cache into the cwd)."""
    cwd = f"{WORK}/run_{name}"
    os.makedirs(cwd, exist_ok=True)
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": gpus, "PYTHONPATH": SCRIPTS, "TQDM_MININTERVAL": "10"}
    if background:
        log = open(f"{cwd}/log.txt", "w")
        return subprocess.Popen(cmd, env=env, cwd=cwd, stdout=log, stderr=subprocess.STDOUT)
    with subprocess.Popen(cmd, env=env, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end="")
    if proc.returncode:
        raise RuntimeError(f"{name} failed with exit code {proc.returncode}; see the output above")


def tail(path, lines=25):
    print("".join(open(path, encoding="utf-8", errors="replace").readlines()[-lines:]))

## 6. Train (and write baseline answers)
The script checks that loss is computed only on the verse answers, then trains and keeps the epoch with the lowest validation loss.

In [ ]:
PY = sys.executable
train_args = [f"{SCRIPTS}/train_shahnameh.py", *DATA_ARGS, "--out", ADAPTER, "--epochs", str(EPOCHS),
              "--lr", str(LEARNING_RATE), "--lora-r", str(LORA_R), "--batch-size", str(BATCH_SIZE),
              "--grad-accum", str(GRAD_ACCUM), "--max-seq-len", str(MAX_SEQ_LEN)]
baseline_cmd = [PY, f"{SCRIPTS}/sample_shahnameh.py", *DATA_ARGS, "--out", f"{WORK}/samples_baseline.jsonl"]

if GPU_MODE == "train+baseline":
    baseline = run(baseline_cmd, "1", "baseline", background=True)
    print("baseline answers are being written on GPU 1 (log: run_baseline/log.txt)")
    try:
        run([PY, *train_args], "0", "train")
    finally:
        code = baseline.wait()
        print(f"--- baseline finished with exit code {code}; last lines of its log:")
        tail(f"{WORK}/run_baseline/log.txt")
    if code:
        raise RuntimeError("baseline sampling failed; see its log above")
else:
    run(baseline_cmd, "0", "baseline")
    run([PY, "-m", "torch.distributed.run", "--nproc_per_node=2", *train_args], "0,1", "train")

## 7. Loss summary
Validation loss should drop. If it rises after the first epoch, the model is memorising: lower `EPOCHS`.

In [ ]:
log = json.load(open(f"{ADAPTER}/train_log.json"))
print("best checkpoint:", log["best_checkpoint"], "| best eval_loss:", log["best_eval_loss"])
for entry in log["log_history"]:
    if "eval_loss" in entry:
        print(f"epoch {entry['epoch']:.2f}: eval_loss {entry['eval_loss']:.4f}")
train_losses = [e["loss"] for e in log["log_history"] if "loss" in e]
print(f"train loss: first {train_losses[0]:.3f}, last {train_losses[-1]:.3f}")

## 8. Answers from the fine-tuned model

In [ ]:
run([sys.executable, f"{SCRIPTS}/sample_shahnameh.py", *DATA_ARGS, "--adapter", ADAPTER,
     "--out", f"{WORK}/samples_finetuned.jsonl"], "0", "finetuned")

## 9. Baseline vs fine-tuned

In [ ]:
from IPython.display import Markdown, display
baseline_rows = common.read_jsonl(f"{WORK}/samples_baseline.jsonl")
finetuned_rows = common.read_jsonl(f"{WORK}/samples_finetuned.jsonl")
report = common.comparison_markdown(baseline_rows, finetuned_rows)
open(f"{WORK}/comparison.md", "w", encoding="utf-8").write(report)
print(f"wrote {WORK}/comparison.md ({len(baseline_rows)} prompts)")
display(Markdown(report))

## 10. Try your own question

In [ ]:
# Ask the fine-tuned model your own question. QUESTION_LANG picks which system prompt is used.
MY_QUESTION = "How do I stay calm before a big exam?"
QUESTION_LANG = "en"   # "fa", "en" or "de"
answer_file = f"{WORK}/custom_answer.jsonl"
run([sys.executable, f"{SCRIPTS}/sample_shahnameh.py", "--preset", MODEL_PRESET, "--data-root", "/kaggle/input",
     "--langs", QUESTION_LANG, *([] if USE_SYSTEM_PROMPT else ["--no-system"]),
     "--adapter", ADAPTER, "--prompt", MY_QUESTION, "--out", answer_file], "0", "custom")
print(common.read_jsonl(answer_file)[0]["answer"])

## 11. Save and export

In [ ]:
import shutil
export_flags = (["--merged"] if EXPORT_MERGED_16BIT else []) + (["--gguf"] if EXPORT_GGUF else [])
if HF_REPO:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    export_flags += ["--hf-repo", HF_REPO]
if export_flags:
    run([sys.executable, f"{SCRIPTS}/export_shahnameh.py", "--preset", MODEL_PRESET, "--adapter", ADAPTER,
         *export_flags], "0", "export")

zip_path = shutil.make_archive(ADAPTER, "zip", ADAPTER)
print("download this from the Output panel:", zip_path)